# Pandas Applying Functions

In [15]:
import pandas as pd

df = pd.read_csv("../Data/data_jobs.csv")
df['job_posted_date'] = pd.to_datetime(df['job_posted_date'], errors='coerce')

In [3]:
help(df.apply)

Help on method apply in module pandas.core.frame:

apply(
    func: 'AggFuncType',
    axis: 'Axis' = 0,
    raw: 'bool' = False,
    result_type: "Literal['expand', 'reduce', 'broadcast'] | None" = None,
    args=(),
    by_row: "Literal[False, 'compat']" = 'compat',
    engine: "Literal['python', 'numba']" = 'python',
    engine_kwargs: 'dict[str, bool] | None' = None,
    **kwargs
) method of pandas.core.frame.DataFrame instance
    Apply a function along an axis of the DataFrame.

    Objects passed to the function are Series objects whose index is
    either the DataFrame's index (``axis=0``) or the DataFrame's columns
    (``axis=1``). By default (``result_type=None``), the final return type
    is inferred from the return type of the applied function. Otherwise,
    it depends on the `result_type` argument.

    Parameters
    ----------
    func : function
        Function to apply to each column or row.
    axis : {0 or 'index', 1 or 'columns'}, default 0
        Axis along wh

* `apply()`: Apply functions to columns or rows.

### Example 1

Calculate projected salaries next year, using an assumed rate of 3.0% for all roles.

In [4]:
def inflation(salary):
    return salary * 1.03

df['salary_year_inflated'] = df['salary_year_avg'].apply(inflation)

df[pd.notna(df['salary_year_avg'])][['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


We can actually simplify this with a lambda function.

In [5]:
df['salary_year_inflated'] = df['salary_year_avg'].apply(lambda salary: salary * 1.03)

df[pd.notna(df['salary_year_avg'])][['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


Now technically this could have been done like this... 

In [6]:
df['salary_year_inflated'] = df['salary_year_avg'] * 1.03

df[pd.notna(df['salary_year_avg'])][['salary_year_avg', 'salary_year_inflated']]

,salary_year_avg,salary_year_inflated
28,109500.0,112785.00
77,140000.0,144200.00
92,120000.0,123600.00
100,228222.0,235068.66
109,89000.0,91670.00
...,...,...
785624,139216.0,143392.48
785641,150000.0,154500.00
785648,221875.0,228531.25
785682,157500.0,162225.00


### Example 2

Calculate projected salaries next year, but:
- For senior roles (e.g., Senior Data Analysts), assume the rate is 5%
- For all other roles, assume rate is 3%

In [7]:
def projected_salary(row):
    if 'Senior' in row['job_title_short']:
        return  1.05 * row['salary_year_avg']
    else:
        return  1.03 * row['salary_year_avg']

df['salary_year_inflated'] = df.apply(projected_salary, axis=1)

df[pd.notna(df['salary_year_avg'])][['job_title_short', 'salary_year_avg', 'salary_year_inflated']]

,job_title_short,salary_year_avg,salary_year_inflated
28,Data Scientist,109500.0,112785.00
77,Data Engineer,140000.0,144200.00
92,Data Engineer,120000.0,123600.00
100,Data Scientist,228222.0,235068.66
109,Data Analyst,89000.0,91670.00
...,...,...,...
785624,Data Engineer,139216.0,143392.48
785641,Data Engineer,150000.0,154500.00
785648,Data Scientist,221875.0,228531.25
785682,Data Scientist,157500.0,162225.00


Technically you could write this with a lambda function:

In [8]:
df['salary_year_inflated'] = df.apply(lambda row: 1.05 * row['salary_year_avg'] if 'Senior' in row['job_title_short'] else 1.03 * row['salary_year_avg'], axis=1)

df[pd.notna(df['salary_year_avg'])][['job_title_short', 'salary_year_avg', 'salary_year_inflated']]

,job_title_short,salary_year_avg,salary_year_inflated
28,Data Scientist,109500.0,112785.00
77,Data Engineer,140000.0,144200.00
92,Data Engineer,120000.0,123600.00
100,Data Scientist,228222.0,235068.66
109,Data Analyst,89000.0,91670.00
...,...,...,...
785624,Data Engineer,139216.0,143392.48
785641,Data Engineer,150000.0,154500.00
785648,Data Scientist,221875.0,228531.25
785682,Data Scientist,157500.0,162225.00


### Example 3

Convert the `job_skills` from a generic object to an actual list object (*hint* this is very important for later). Let's try doing that by just using `ast.literal_eval` and then look at our new column.

A reminder of what our `job_skills` column looks like now:

In [16]:
df['job_skills']

0         ['r', 'python', 'sql', 'nosql', 'power bi', 't...
1         ['python', 'sql', 'c#', 'azure', 'airflow', 'd...
2         ['python', 'c++', 'java', 'matlab', 'aws', 'te...
3         ['bash', 'python', 'oracle', 'aws', 'ansible',...
4                                  ['python', 'sql', 'gcp']
                                ...                        
785735    ['bash', 'python', 'perl', 'linux', 'unix', 'k...
785736                       ['sas', 'sas', 'sql', 'excel']
785737                              ['powerpoint', 'excel']
785738    ['python', 'go', 'nosql', 'sql', 'mongo', 'she...
785739                                      ['aws', 'flow']
Name: job_skills, Length: 785740, dtype: object

In [17]:
df['job_skills'][0]

"['r', 'python', 'sql', 'nosql', 'power bi', 'tableau']"

In [18]:
type(df['job_skills'][0])

str

In [19]:
import ast

ast.literal_eval(df['job_skills'][0])

['r', 'python', 'sql', 'nosql', 'power bi', 'tableau']

In [20]:
type(ast.literal_eval(df['job_skills'][0]))

list

In [21]:
import ast

# Convert string representation to actual list, checking for NaN values first
df['job_skills'] = df['job_skills'].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else x)

In [22]:
df['job_skills']

0                [r, python, sql, nosql, power bi, tableau]
1         [python, sql, c#, azure, airflow, dax, docker,...
2         [python, c++, java, matlab, aws, tensorflow, k...
3         [bash, python, oracle, aws, ansible, puppet, j...
4                                        [python, sql, gcp]
                                ...                        
785735    [bash, python, perl, linux, unix, kubernetes, ...
785736                               [sas, sas, sql, excel]
785737                                  [powerpoint, excel]
785738    [python, go, nosql, sql, mongo, shell, mysql, ...
785739                                          [aws, flow]
Name: job_skills, Length: 785740, dtype: object

### Problems

> Convert Date to String

In [31]:
df['job_posted_date_str'] = df['job_posted_date'].apply(lambda date: date.strftime('%Y-%m-%d'))
df[['job_posted_date', 'job_posted_date_str']].head()

,job_posted_date,job_posted_date_str
0,2023-01-14 13:18:07,2023-01-14
1,2023-10-10 13:14:55,2023-10-10
2,2023-07-04 13:01:41,2023-07-04
3,2023-08-07 14:29:36,2023-08-07
4,2023-11-07 14:01:59,2023-11-07


> Days Since Posted

In [46]:
from datetime import datetime

current_date = pd.Timestamp(datetime.now())

#df['days_since_posted'] = abs(df['job_posted_date'] - current_date).dt.days
df['job_posted_date'].apply(lambda date: abs(date - current_date).days)

0         1099
1          830
2          928
3          894
4          802
          ... 
785735    1041
785736    1042
785737    1042
785738    1042
785739    1041
Name: job_posted_date, Length: 785740, dtype: int64

> Salary Category

In [47]:
df_filtered = df.dropna(subset='salary_year_avg').copy()
df_filtered.head()

,job_title_short,job_title,job_location,job_via,job_schedule_type,job_work_from_home,search_location,job_posted_date,job_no_degree_mention,job_health_insurance,job_country,salary_rate,salary_year_avg,salary_hour_avg,company_name,job_skills,job_type_skills,job_posted_date_str
27,Data Scientist,CRM Data Specialist,"San José Province, San José, Costa Rica",via Ai-Jobs.net,Full-time,False,Costa Rica,2023-08-01 13:37:57,False,False,Costa Rica,year,109500.0,NaN,Netskope,"[gdpr, excel]","{'analyst_tools': ['excel'], 'libraries': ['gd...",2023-08-01
76,Data Engineer,Data Engineer,"Arlington, VA",via LinkedIn,Full-time,False,Sudan,2023-06-26 14:22:54,False,False,Sudan,year,140000.0,NaN,Intelletec,"[mongodb, mongodb, python, r, sql, mysql, mari...","{'analyst_tools': ['tableau'], 'cloud': ['orac...",2023-06-26
91,Data Engineer,Remote - Data Engineer - Permanent - W2,Anywhere,via LinkedIn,Full-time,True,"Illinois, United States",2023-02-21 13:29:59,False,True,United States,year,120000.0,NaN,Apex Systems,"[sql, python]","{'programming': ['sql', 'python']}",2023-02-21
99,Data Scientist,"Data Scientist, Risk Data Mining - USDS","Mountain View, CA",via LinkedIn,Full-time,False,"California, United States",2023-07-31 13:01:18,False,True,United States,year,228222.0,NaN,TikTok,"[sql, r, python, express]","{'programming': ['sql', 'r', 'python'], 'webfr...",2023-07-31
108,Data Analyst,Senior Supply Chain Analytics Analyst,Anywhere,via Get.It,Full-time,True,"Illinois, United States",2023-10-12 13:02:19,False,True,United States,year,89000.0,NaN,Get It Recruit - Transportation,"[python, r, alteryx, tableau]","{'analyst_tools': ['alteryx', 'tableau'], 'pro...",2023-10-12


In [55]:
df_filtered['salary_category'] = df_filtered['salary_year_avg'].apply(lambda salary: 'Low' if salary < 60000 else 'Medium' if salary <= 100000 else 'High' )

In [56]:
df_filtered[['salary_year_avg', 'salary_category']]

,salary_year_avg,salary_category
27,109500.0,High
76,140000.0,High
91,120000.0,High
99,228222.0,High
108,89000.0,Medium
...,...,...
785623,139216.0,High
785640,150000.0,High
785647,221875.0,High
785681,157500.0,High
